# House Stage 1: ablations, rolling backtests

The single-ablation notebooks, merged (2026-11-03) with their cells unchanged; each section is self-contained and redefines its own settings, so they run in order after the shared setup below. Their decisions are the ones in `src/house/stage1.py`.

1. **Fixed incumbency coefficient vs OLS** (was `01_reg_2018to2022_fixed_inc.ipynb`)
2. **WLS weightings vs OLS** (was `01_reg_2018to2022_wls.ipynb`)
3. **No generic ballot** (was `01_reg_no_generic_ballot.ipynb`)
4. **Model B without partisan lean in the ensemble** (was `01_reg_ensemble_nolean_b.ipynb`)

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import Markdown, display
from scipy import stats

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from house.experiments import evaluation, tracking
from house.experiments.run import run_ols, run_ols_fixed, run_wls, training_weights
from house import stage1
from house.regression.schema import KEY

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

## 1. Fixed incumbency coefficient vs OLS


The classic formula with the `inc_signed` coefficient **imposed** rather than estimated, against unrestricted OLS, in the same expanding-window backtest as `01_regression_2018to2022_eval.ipynb`. Each window refits every model on prior elections only and scores the next one:

| eval year | train years |
|---|---|
| 2018 | 2006–2016 |
| 2020 | 2006–2018 |
| 2022 | 2006–2020 |

For a fixed coefficient *c*, `run_ols_fixed` (`src/house/experiments/run.py`) fits OLS of `adj_margin_2p_norm − c · inc_signed` on the other terms, re-estimating all of them, then adds `c · inc_signed` back to the fitted values and predictions. The predictions are scored against `adj_margin_2p_norm` as usual. Every model in a window uses the same training and evaluation rows, and the evaluation subsets (valid, `COMPETITIVE`) are the original notebook's. 2024 is never used (`USE_HOLDOUT = False`).

In [ ]:
FORMULA = "adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed"
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
REQUIRE_COMPLETE = []   # extra columns a row must have; every model below uses FORMULA's columns, so the same rows
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

### Fixed incumbency coefficients

`inc_signed` is +1 for a Democratic incumbent, −1 for a Republican one and 0 for an open seat, so its coefficient is the incumbency advantage in points of D margin. Each value in `FIXED_INC_COEFFICIENTS` is imposed in every window. Unrestricted OLS (`ols`) estimates it freely and is the base of every Δ below. Fixing it at 0 is the same as dropping the term.

In [ ]:
FIXED_TERM = "inc_signed"
FIXED_INC_COEFFICIENTS = [0, 2.5, 5.0, 7.5, 10.0]  # imposed coefficients, points of D margin

FIXED_VARIANTS = {f"inc_fixed_{c:g}": c for c in FIXED_INC_COEFFICIENTS}  # variant: its imposed coefficient
list(FIXED_VARIANTS)

In [ ]:
# Every window: unrestricted OLS (the base), then each fixed coefficient, all on the same rows
rolling = {}
for eval_year, train_years in ROLLING_WINDOWS.items():
    ols = run_ols(
        formula=FORMULA,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=REQUIRE_COMPLETE,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name="ols",
        save=False,
    )
    fixed = [
        run_ols_fixed(
            formula=FORMULA,
            train_years=train_years,
            eval_years=[eval_year],
            predict_years=[],
            fixed={FIXED_TERM: c},
            predictors=predictors,
            targets=targets,
            use_holdout=USE_HOLDOUT,
            require_complete=REQUIRE_COMPLETE,
            race_types=EVAL_RACE_TYPES,
            competitive=COMPETITIVE,
            name=name,
            save=False,
        )
        for name, c in FIXED_VARIANTS.items()
    ]
    rolling[eval_year] = {"ols_vs_fixed": [ols, *fixed]}  # one block, OLS first: the base of every delta

In [ ]:
# the regression each variant actually fits (the same in every window)
pd.Series({run.config["name"]: run.config.get("fit_formula", run.config["formula"])
           for run in rolling[next(iter(ROLLING_WINDOWS))]["ols_vs_fixed"]}, name="fit formula").to_frame()

### Results

For each section (ALL, EX_ANTE and EX_POST on the original notebook's definitions), there are two tables. The first is **by eval year**, each window on its own. The second is the **mean over 2018, 2020 and 2022**, each window weighted equally. Variants are ranked by mean MAE, with ties broken by RMSE.

- **Metrics:** `n_train`, `n`, MAE, RMSE, bias (mean residual, actual − predicted: + means Democrats outperformed the forecast), accuracy, precision D/R and misses.
- **Mean table extras:** `|bias|` (the mean of each window's |bias|, so opposite misses don't cancel) and `mae_<year>` for each window.
- **Δ columns:** variant − unrestricted OLS in the same window, on the same rows. ✓ (green) marks an improvement and ✗ (red) a regression: lower MAE, RMSE, |bias| and misses, higher accuracy and precision.

In [ ]:
VARIANTS = len(FIXED_VARIANTS)  # show every variant, ranked

for section, subset in SUMMARY_SECTIONS.items():
    by_year = pd.concat(
        {year: tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)[0]
         for year, blocks in rolling.items()}, names=["eval_year"]).droplevel("block")
    mean, _ = tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
    display(Markdown(f"### {section} (`{subset}`): by eval year"))
    display(tracking.style_best_of(by_year))
    display(Markdown(f"### {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}, "
                     f"ranked by {RANK_BY} (ties: {TIEBREAK})"))
    display(tracking.style_best_of(mean.droplevel("block")))

### Coefficients

Each model's coefficients by window, then averaged over the three. For the fixed variants, `inc_signed` is the imposed value (it has no standard error) and the other terms are re-estimated around it. Unrestricted OLS's estimate is the `ols` row.

In [ ]:
# coefficients by window (rows: eval year and model, columns: terms); inc_signed is imposed for the fixed variants
pd.concat({eval_year: pd.DataFrame({run.config["name"]: run.coefficients["coef"] for run in blocks["ols_vs_fixed"]}).T
           for eval_year, blocks in rolling.items()}, names=["eval_year", "model"]).round(3)

In [ ]:
# each model's coefficients averaged over the windows
_, mean_coefficients = tracking.rolling_best_of(rolling, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
mean_coefficients.droplevel("block").round(3)

### Ablations with incumbency fixed

The ablation blocks of `01_regression_2018to2022_eval.ipynb`, refit with `inc_signed` fixed at `ABLATION_FIXED_INC` (2.5) instead of estimated: demographics, interactions, FEC fundraising, small-dollar fundraising, and the relative economic factors. Each model is `run_ols_fixed(..., fixed={"inc_signed": ABLATION_FIXED_INC})`, so every other coefficient, including each block's added terms, is re-estimated around the fixed incumbency effect. Each block is refit in every window (`ROLLING_WINDOWS`) on prior elections only.

- **Bases:** each block's base is the classic formula with incumbency fixed, and its `Δ` columns compare against that base, on the same rows. A block shares its rows across its models through `require_complete`, every predictor column any of its formulas uses. A block's base is shown separately when its rows differ from the first block's (FEC leaves out the districts it has no data for; small-dollar trains from 2008).
- **Left out:** variants that estimate incumbency freely (`no_inc_signed`, `inc_categorical`, `plus_fundraising_no_inc`) conflict with a fixed coefficient. Fixing it at 0 is the `inc_fixed_0` row above.
- **Interactions** such as `partisan_lean:inc_signed` are estimated on top of the fixed main effect.
- **Economic factors** are the relative indicators (region or state against the nation), each signed by the president's party (`pres_party_signed:<variable>`, as in the original notebook). β is then the effect on the president's party's margin.

The tables are by eval year and mean over the windows, with the same metrics, Δ and ✓ / ✗ marks as above. `ABLATION_TOP` limits each block to its best variants by mean MAE (ties: RMSE). `None` shows every variant.

In [ ]:
ABLATION_FIXED_INC = 2.5  # inc_signed imposed in every ablation, points of D margin
ABLATION_TOP = None       # variants shown per block, ranked; None shows all, 2 the top two
ABLATION_BASE = f"inc_fixed_{ABLATION_FIXED_INC:g}"
TARGET = "adj_margin_2p_norm"

# The party holding the White House at each election: +1 Democratic, -1 Republican (the economic factors' sign)
PRES_PARTY_SIGNED = {2006: -1, 2008: -1, 2010: 1, 2012: 1, 2014: 1, 2016: 1, 2018: -1, 2020: -1, 2022: 1, 2024: 1,
                     2026: -1}
predictors = predictors.assign(pres_party_signed=predictors["year"].map(PRES_PARTY_SIGNED))

DEMOGRAPHIC_ABLATIONS = {
    ABLATION_BASE: FORMULA,  # the base
    "plus_acs": FORMULA + " + college + hispanic",
}
INTERACTION_ABLATIONS = {
    ABLATION_BASE: FORMULA,
    "lean_x_inc": FORMULA + " + partisan_lean:inc_signed",
}
FEC_ABLATIONS = {
    ABLATION_BASE: FORMULA,
    "plus_fundraising": FORMULA + " + fundraising_advantage",
    "plus_spending": FORMULA + " + spending_advantage",
    "plus_fundraising_spending": FORMULA + " + fundraising_advantage + spending_advantage",
}
SMALL_DOLLAR_ABLATIONS = {  # trained from 2008: small_dollar_share_diff has no 2006 data
    f"{ABLATION_BASE}_from_2008": FORMULA,
    "plus_fundraising_from_2008": FORMULA + " + fundraising_advantage",
    "plus_fundraising_small_dollar": FORMULA + " + fundraising_advantage + small_dollar_share_diff",
}
RELATIVE_ECON_ABLATIONS = {  # the region or state against the nation, signed by the president's party
    ABLATION_BASE: FORMULA,
    "pres_x_cpi_region_minus_us_yoy": FORMULA + " + pres_party_signed:cpi_region_minus_us_yoy",  # pp, inflation
    "pres_x_gas_region_minus_us": FORMULA + " + pres_party_signed:gas_region_minus_us",          # $ gap, gas price
    "pres_x_log_rel_gas_price": FORMULA + " + pres_party_signed:log_rel_gas_price",              # log ratio, gas
    "pres_x_gas_region_minus_us_yoy": FORMULA + " + pres_party_signed:gas_region_minus_us_yoy",  # relative trend
    "pres_x_unemp_state_minus_us": FORMULA + " + pres_party_signed:unemp_state_minus_us",        # pp, unemployment
    "pres_x_unemp_change_12m_minus_us": FORMULA + " + pres_party_signed:unemp_change_12m_minus_us",  # relative trend
}

ABLATION_BLOCKS = {  # block: (formulas, base first; first training year)
    "demographics": (DEMOGRAPHIC_ABLATIONS, 2006),
    "interactions": (INTERACTION_ABLATIONS, 2006),
    "fec": (FEC_ABLATIONS, 2006),
    "small_dollar": (SMALL_DOLLAR_ABLATIONS, 2008),
    "relative_econ": (RELATIVE_ECON_ABLATIONS, 2006),
}
# each block's shared rows: every predictor column any of its formulas uses
ABLATION_SHARED_COLUMNS = {block: evaluation.shared_columns(formulas=formulas.values(), columns=predictors.columns)
                           for block, (formulas, _) in ABLATION_BLOCKS.items()}
ABLATION_SHARED_COLUMNS

In [ ]:
# the missing-predictor warnings (FEC has no data for a few winners) repeat for every window and model; each run's
# exclusions are in its .excluded and .counts
with warnings.catch_warnings(action="ignore"):
    ablations = {
        eval_year: {
            block: [
                run_ols_fixed(
                    formula=formula,
                    train_years=[year for year in train_years if year >= first_train_year],
                    eval_years=[eval_year],
                    predict_years=[],
                    fixed={FIXED_TERM: ABLATION_FIXED_INC},
                    predictors=predictors,
                    targets=targets,
                    use_holdout=USE_HOLDOUT,
                    require_complete=ABLATION_SHARED_COLUMNS[block],
                    race_types=EVAL_RACE_TYPES,
                    competitive=COMPETITIVE,
                    name=name,
                    save=False,
                )
                for name, formula in formulas.items()
            ]
            for block, (formulas, first_train_year) in ABLATION_BLOCKS.items()
        }
        for eval_year, train_years in ROLLING_WINDOWS.items()
    }

In [ ]:
for section, subset in SUMMARY_SECTIONS.items():
    by_year = pd.concat(
        {year: tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=ABLATION_TOP, tiebreak=TIEBREAK)[0]
         for year, blocks in ablations.items()}, names=["eval_year"])
    mean, _ = tracking.rolling_best_of(ablations, subset=subset, rank_by=RANK_BY, top=ABLATION_TOP,
                                       tiebreak=TIEBREAK)
    display(Markdown(f"### Ablations, {section} (`{subset}`), inc_signed fixed at {ABLATION_FIXED_INC:g}: "
                     "by eval year"))
    display(tracking.style_best_of(by_year))
    display(Markdown(f"### Ablations, {section} (`{subset}`), inc_signed fixed at {ABLATION_FIXED_INC:g}: mean of "
                     f"{', '.join(map(str, ROLLING_WINDOWS))}, ranked by {RANK_BY} (ties: {TIEBREAK})"))
    display(tracking.style_best_of(mean))

In [ ]:
# each ablation's coefficients averaged over the windows; inc_signed is the imposed ABLATION_FIXED_INC throughout.
# The pres_party_signed:<variable> rows are effects on the president's party's margin, per unit of the variable
_, ablation_coefficients = tracking.rolling_best_of(ablations, rank_by=RANK_BY, top=ABLATION_TOP, tiebreak=TIEBREAK)
ablation_coefficients.style.format(precision=3, na_rep="-")

## 2. WLS weightings vs OLS


Weighted least squares against OLS on the classic formula, in the same expanding-window backtest as `01_regression_2018to2022_eval.ipynb`. Each window refits every model on its own training years and scores the next election:

| eval year | train years |
|---|---|
| 2018 | 2006–2016 |
| 2020 | 2006–2018 |
| 2022 | 2006–2020 |

`run_wls` (`src/house/experiments/run.py`) is `run_ols` with one change: the fit is `smf.wls(formula, data=train, weights=training_weights(train, ...)).fit()`. The weights come from the training rows alone, after exclusions, and are recomputed and normalized to mean 1 in every window, so nothing from the eval year enters a fit. Every model in a window uses the same training and evaluation rows, and the evaluation subsets (valid, `COMPETITIVE`) are the original notebook's. 2024 is never used (`USE_HOLDOUT = False`).

In [ ]:
FORMULA = "adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed"
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
REQUIRE_COMPLETE = []   # extra columns a row must have; every model below uses the same formula, so the same rows
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

### Weighting

`training_weights` multiplies three factors for a training row from election year *t*. Each is on or off per variant:

- **Equal-year:** w = 1 / n_t, where n_t is the number of valid training rows in year *t*. Every training year then carries the same total weight, whatever its number of usable districts.
- **Recency:** w = 2^(−(T − t)/H), where T is the window's latest training year. A year H years older counts half. Measuring from the eval year instead only rescales the weights, which normalization removes.
- **Election type:** w = `MIDTERM_WEIGHT` in midterm years (2006, 2010, …) and 1.0 in presidential years.

The weights are then normalized to mean 1 within the window. Rescaling doesn't change WLS estimates. Every variant is compared with OLS (all weights 1), which is the base of every Δ below.

In [ ]:
HALF_LIVES = [6, 8, 12]  # recency half-lives, in years
MIDTERM_WEIGHT = 1.25    # midterm elections, against 1.0 for presidential ones

WLS_WEIGHTINGS = {  # variant: training_weights settings
    "wls_equal_year": {"equal_year": True, "half_life": None, "midterm_weight": 1.0},
    **{f"wls_recency_h{h}": {"equal_year": False, "half_life": h, "midterm_weight": 1.0} for h in HALF_LIVES},
    **{f"wls_equal_year_recency_h{h}": {"equal_year": True, "half_life": h, "midterm_weight": 1.0}
       for h in HALF_LIVES},
    "wls_election_type": {"equal_year": False, "half_life": None, "midterm_weight": MIDTERM_WEIGHT},
    **{f"wls_all_h{h}": {"equal_year": True, "half_life": h, "midterm_weight": MIDTERM_WEIGHT} for h in HALF_LIVES},
}
list(WLS_WEIGHTINGS)

In [ ]:
# Every window: OLS (the base), then each WLS variant, all on the same rows
rolling = {}
for eval_year, train_years in ROLLING_WINDOWS.items():
    ols = run_ols(
        formula=FORMULA,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=REQUIRE_COMPLETE,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name="ols",
        save=False,
    )
    wls = [
        run_wls(
            formula=FORMULA,
            train_years=train_years,
            eval_years=[eval_year],
            predict_years=[],
            equal_year=weights["equal_year"],
            half_life=weights["half_life"],
            midterm_weight=weights["midterm_weight"],
            predictors=predictors,
            targets=targets,
            use_holdout=USE_HOLDOUT,
            require_complete=REQUIRE_COMPLETE,
            race_types=EVAL_RACE_TYPES,
            competitive=COMPETITIVE,
            name=name,
            save=False,
        )
        for name, weights in WLS_WEIGHTINGS.items()
    ]
    rolling[eval_year] = {"ols_vs_wls": [ols, *wls]}  # one block, OLS first: the base of every delta

In [ ]:
# Each training year's share of the total weight (%), in each window: what every variant actually does to the sample
shares = {}
for eval_year, blocks in rolling.items():
    train = blocks["ols_vs_wls"][0].predictions.query("role == 'train'")[["year"]]
    for name, weights in {"ols": {}, **WLS_WEIGHTINGS}.items():
        share = training_weights(train, **weights).groupby(train["year"]).sum()
        shares[(eval_year, name)] = 100 * share / share.sum()
pd.DataFrame(shares).rename_axis(columns=["eval_year", "variant"]).round(1)

### Results

For each section (ALL, EX_ANTE and EX_POST on the original notebook's definitions), there are two tables. The first is **by eval year**, each window on its own. The second is the **mean over 2018, 2020 and 2022**, each window weighted equally. Variants are ranked by mean MAE, with ties broken by RMSE.

- **Metrics:** `n_train`, `n`, MAE, RMSE, bias (mean residual, actual − predicted: + means Democrats outperformed the forecast), accuracy, precision D/R and misses.
- **Mean table extras:** `|bias|` (the mean of each window's |bias|, so opposite misses don't cancel) and `mae_<year>` for each window.
- **Δ columns:** variant − OLS in the same window, on the same rows. ✓ (green) marks an improvement and ✗ (red) a regression: lower MAE, RMSE, |bias| and misses, higher accuracy and precision.

In [ ]:
VARIANTS = len(WLS_WEIGHTINGS)  # show every variant, ranked

for section, subset in SUMMARY_SECTIONS.items():
    by_year = pd.concat(
        {year: tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)[0]
         for year, blocks in rolling.items()}, names=["eval_year"]).droplevel("block")
    mean, _ = tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
    display(Markdown(f"### {section} (`{subset}`): by eval year"))
    display(tracking.style_best_of(by_year))
    display(Markdown(f"### {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}, "
                     f"ranked by {RANK_BY} (ties: {TIEBREAK})"))
    display(tracking.style_best_of(mean.droplevel("block")))

### Coefficients

The fitted coefficients of every model, by window and averaged over the three. Weighting moves the estimates through how much each training year counts. Incumbency (`inc_signed`) is shown on its own, with its change from OLS.

In [ ]:
# each model's coefficients averaged over the windows (rows: models, columns: terms)
_, mean_coefficients = tracking.rolling_best_of(rolling, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
mean_coefficients.droplevel("block").round(3)

In [ ]:
# the incumbency coefficient (points of margin per unit of inc_signed: +1 D incumbent, -1 R, 0 open), by window
incumbency = pd.DataFrame({
    eval_year: {run.config["name"]: run.coefficients.loc["inc_signed", "coef"] for run in blocks["ols_vs_wls"]}
    for eval_year, blocks in rolling.items()
}).rename_axis(index="model", columns="eval_year")
incumbency["mean"] = incumbency.mean(axis=1)
incumbency["Δmean vs ols"] = incumbency["mean"] - incumbency.loc["ols", "mean"]
incumbency.round(3)

## 3. No generic ballot


District margins forecast **without the generic ballot**: from `partisan_lean`, incumbency, and a small, theory-driven set of fundamentals, in the same expanding-window backtest as the other notebooks:

| eval year | train years |
|---|---|
| 2018 | 2006–2016 |
| 2020 | 2006–2018 |
| 2022 | 2006–2020 |

**No generic-ballot data enters any no-GB model**, in fitting or in prediction. Their formulas and their `require_complete` rows use no `gen_margin_raw` or `gb_*` column, which the guard cell below asserts. The ex-ante competitiveness filter (|`partisan_lean` + `gb_margin_2p_norm`| ≤ 10) uses the generic ballot, but only to choose which districts are scored, never in a model. The ex-post filter (|margin| ≤ 7.5) is unchanged. 2024 is never used (`USE_HOLDOUT = False`).

**Incumbency:** `inc_signed` unrestricted (OLS), or fixed at 2.5 or 5.0 (`run_ols_fixed`: OLS of the target minus `c · inc_signed`, with it added back to the predictions), each with and without `partisan_lean:inc_signed`.

**Additional predictors,** each added on its own to every base:

- **ACS:** `rel_state_college` (district − state college share, points) and `hispanic`, tested separately.
- **FEC:** `fundraising_advantage`. This is exploratory: historical totals run through December 31, after the election.
- **Economy,** from September of the election year, the last month published before Election Day:
  - `unemp_change_12m` (state, 12-month change)
  - `unemp_state_minus_us` (state − national rate)
  - `cpi_us_yoy` (national inflation)
  - `cpi_region_minus_us_yoy` (regional − national inflation)
  - `log_gas_yoy` (regional gasoline, 12-month log change)

  Each enters signed by the president's party (`pres_party_signed:<variable>`, +1 for a Democratic president, −1 for a Republican, known before every election), so its β is the effect on the president's party's margin. Without the generic ballot, these signed terms are the only way the national environment can enter these models.

**Two economic combinations,** and no more:

- **National economy:** the state's 12-month unemployment change + national inflation, the two halves of a misery-index judgment of the president.
- **Local economy:** unemployment and inflation relative to the nation.

Gasoline is not combined with CPI, since gasoline prices are part of the CPI. The ACS and FEC terms are not combined with the economy, which keeps the models from becoming kitchen sinks.

**Comparison with the generic-ballot finalists:** the same 11 feature sets on `GB_FORMULA` (with the generic ballot) with incumbency fixed at 2.5, with and without the interaction. `gb_inc_fixed_2.5_lean_x_inc`'s base is the finalists notebook's base model. Every model, with or without the generic ballot, scores exactly the same rows.

**Caveats:**

- **National-only variables:** `cpi_us_yoy` is the same for every district in a year, so its effect is estimated from 6–8 training years. 2022's 8.2% is far above every training year (1.1–4.9%), so it extrapolates.
- **Optimistic picks:** any feature set picked from these three eval years scores optimistically.

**Priority:** ex-ante MAE and winner accuracy first. ALL and ex-post metrics are diagnostics.

In [ ]:
TARGET = "adj_margin_2p_norm"
NO_GB_FORMULA = f"{TARGET} ~ partisan_lean + inc_signed"              # no generic ballot
GB_FORMULA = f"{TARGET} ~ gen_margin_raw + partisan_lean + inc_signed"  # the classic formula, for comparison
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

### Models

Each block is one incumbency treatment on one base formula, and holds the same 11 feature sets (`FEATURE_SETS`). Its first model, `base`, is the base of every Δ in the block. For unrestricted OLS, `fixed={}`: `run_ols_fixed` with nothing fixed is `run_ols`.

In [ ]:
FIXED_TERM = "inc_signed"
LEAN_X_INC = " + partisan_lean:inc_signed"

# The party holding the White House at each election, known before it: +1 Democratic, -1 Republican
PRES_PARTY_SIGNED = {2006: -1, 2008: -1, 2010: 1, 2012: 1, 2014: 1, 2016: 1, 2018: -1, 2020: -1, 2022: 1, 2024: 1,
                     2026: -1}
predictors = predictors.assign(pres_party_signed=predictors["year"].map(PRES_PARTY_SIGNED))

FEATURE_SETS = {  # model: the terms it adds to its block's base formula
    "base": [],
    "plus_rel_state_college": ["rel_state_college"],
    "plus_hispanic": ["hispanic"],
    "plus_fundraising": ["fundraising_advantage"],  # exploratory: post-election money in historical totals
    "pres_x_unemp_change_12m": ["pres_party_signed:unemp_change_12m"],
    "pres_x_unemp_state_minus_us": ["pres_party_signed:unemp_state_minus_us"],
    "pres_x_cpi_us_yoy": ["pres_party_signed:cpi_us_yoy"],
    "pres_x_cpi_region_minus_us_yoy": ["pres_party_signed:cpi_region_minus_us_yoy"],
    "pres_x_log_gas_yoy": ["pres_party_signed:log_gas_yoy"],
    # unemployment change + inflation, nationally judged
    "national_economy": ["pres_party_signed:unemp_change_12m", "pres_party_signed:cpi_us_yoy"],
    # unemployment and inflation relative to the nation
    "local_economy": ["pres_party_signed:unemp_state_minus_us", "pres_party_signed:cpi_region_minus_us_yoy"],
}
NO_GB_INCUMBENCY = {"ols": None, "inc_fixed_2.5": 2.5, "inc_fixed_5": 5.0}  # treatment: fixed coefficient (None: OLS)
GB_INCUMBENCY = {"inc_fixed_2.5": 2.5}  # the generic-ballot comparison: the finalists' fixed coefficient

BLOCKS = {  # block: (base formula, fixed inc_signed coefficient or None)
    **{f"no_gb_{name}{suffix}": (NO_GB_FORMULA + interaction, c)
       for name, c in NO_GB_INCUMBENCY.items() for suffix, interaction in [("", ""), ("_lean_x_inc", LEAN_X_INC)]},
    **{f"gb_{name}{suffix}": (GB_FORMULA + interaction, c)
       for name, c in GB_INCUMBENCY.items() for suffix, interaction in [("", ""), ("_lean_x_inc", LEAN_X_INC)]},
}
GB_FINALIST = ("gb_inc_fixed_2.5_lean_x_inc", "base")  # the generic-ballot finalist base: the reference below


def with_terms(base_formula, terms):
    return base_formula + "".join(f" + {term}" for term in terms)


pd.Series({block: base for block, (base, _) in BLOCKS.items()}, name="base formula").to_frame().assign(
    fixed_inc=[c for _, c in BLOCKS.values()])

In [ ]:
# Guard: no generic-ballot column in any no-GB model's formula or rows; then each family's shared rows
GB_COLUMNS = ["gen_margin_raw", *(column for column in predictors.columns if column.startswith("gb_"))]
NO_GB_FORMULAS = [with_terms(base, terms) for block, (base, _) in BLOCKS.items() if block.startswith("no_gb_")
                  for terms in FEATURE_SETS.values()]
GB_FORMULAS = [with_terms(base, terms) for block, (base, _) in BLOCKS.items() if block.startswith("gb_")
               for terms in FEATURE_SETS.values()]
for formula in NO_GB_FORMULAS:
    _, variables = evaluation.formula_variables(formula, predictors.columns)
    assert not set(variables) & set(GB_COLUMNS), f"generic-ballot column in a no-GB model: {formula}"

SHARED_COLUMNS = {  # require_complete: every column any model of the family uses
    "no_gb": evaluation.shared_columns(formulas=NO_GB_FORMULAS, columns=predictors.columns),
    "gb": evaluation.shared_columns(formulas=GB_FORMULAS, columns=predictors.columns),
}
assert not set(SHARED_COLUMNS["no_gb"]) & set(GB_COLUMNS)
SHARED_COLUMNS

In [ ]:
# Collinearity of the candidate terms, on the latest window's training rows (2006-2020): the combinations pair terms
# that aren't near-duplicates
train = predictors[predictors["year"].isin(ROLLING_WINDOWS[max(ROLLING_WINDOWS)])]
terms = pd.DataFrame({
    "partisan_lean": train["partisan_lean"],
    "inc_signed": train["inc_signed"],
    "rel_state_college": train["rel_state_college"],
    "hispanic": train["hispanic"],
    "fundraising_advantage": train["fundraising_advantage"],
    **{f"pres_x_{v}": train["pres_party_signed"] * train[v]
       for v in ["unemp_change_12m", "unemp_state_minus_us", "cpi_us_yoy", "cpi_region_minus_us_yoy", "log_gas_yoy"]},
})
terms.corr().round(2).style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1).format(precision=2)

In [ ]:
# Every window: every block's 11 models. The missing-predictor warnings (FEC has no data for a few winners) repeat for
# every window and model; each run's exclusions are in its .excluded and .counts
with warnings.catch_warnings(action="ignore"):
    rolling = {
        eval_year: {
            block: [
                run_ols_fixed(
                    formula=with_terms(base_formula, terms),
                    train_years=train_years,
                    eval_years=[eval_year],
                    predict_years=[],
                    fixed={} if fixed_inc is None else {FIXED_TERM: fixed_inc},
                    predictors=predictors,
                    targets=targets,
                    use_holdout=USE_HOLDOUT,
                    require_complete=SHARED_COLUMNS["no_gb" if block.startswith("no_gb_") else "gb"],
                    race_types=EVAL_RACE_TYPES,
                    competitive=COMPETITIVE,
                    name=name,
                    save=False,
                )
                for name, terms in FEATURE_SETS.items()
            ]
            for block, (base_formula, fixed_inc) in BLOCKS.items()
        }
        for eval_year, train_years in ROLLING_WINDOWS.items()
    }

# matched samples: in every window, every model (with and without the generic ballot) trains and scores the same rows
for eval_year, blocks in rolling.items():
    rows = {(run.counts["train_rows_hash"], run.counts["eval_rows_hash"]) for runs in blocks.values() for run in runs}
    assert len(rows) == 1, f"{eval_year}: models use different rows"
print(f"{sum(len(runs) for runs in rolling[max(rolling)].values())} models per window, all on the same rows")

### Overview: every model by block

Each grid is a mean over 2018, 2020 and 2022, each weighted equally. Rows are the feature sets and columns are the blocks; the best value in each column is bold.

- **Ex-ante MAE and winner accuracy come first.** ALL and ex-post follow as diagnostics.
- **Against the generic-ballot finalist base:** the last grids compare every model with `GB_FINALIST`, the finalists notebook's base. All models score the same rows, so these differences are matched.

In [ ]:
def mean_table(subset):
    """Every model's metrics on `subset`, averaged over the windows, indexed by (block, model)."""
    return tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)[0]


MEANS = {section: mean_table(subset) for section, subset in SUMMARY_SECTIONS.items()}


def grid(section, metric):
    """`metric` as a feature set x block grid."""
    return MEANS[section][metric].unstack("block").reindex(index=list(FEATURE_SETS), columns=list(BLOCKS))


GRIDS = [  # (section, metric, lower is better), in priority order
    ("EX_ANTE_COMPETITIVE", "mae", True),
    ("EX_ANTE_COMPETITIVE", "accuracy_pct", False),
    ("ALL", "mae", True),
    ("ALL", "accuracy_pct", False),
    ("EX_POST_COMPETITIVE", "mae", True),
    ("EX_POST_COMPETITIVE", "accuracy_pct", False),
]
for section, metric, lower in GRIDS:
    table = grid(section, metric)
    display(Markdown(f"### {section}: mean {metric}"))
    display((table.style.highlight_min if lower else table.style.highlight_max)(axis=0, props="font-weight: bold")
            .format(precision=1 if metric.endswith("_pct") else 2))

In [ ]:
# Every model against the generic-ballot finalist base (matched rows): negative ΔMAE and positive Δaccuracy are better
for section, metric, lower in GRIDS[:2]:
    delta = grid(section, metric) - MEANS[section].loc[GB_FINALIST, metric]
    display(Markdown(f"### {section}: Δ{metric} vs `{GB_FINALIST[0]}` / `{GB_FINALIST[1]}`"))
    display(delta.style.format("{:+.2f}" if lower else "{:+.1f}")
            .map(lambda v: "" if pd.isna(v) or v == 0 else
                 f"color: {tracking.GOOD if (v < 0) == lower else tracking.BAD}"))

### Detail by section

Each block's base and its `DETAIL_TOP` best models by mean MAE (ties: RMSE), with every metric, the per-window MAE, and Δ against the block's base. `None` shows every model.

In [ ]:
DETAIL_TOP = 3

for section, subset in SUMMARY_SECTIONS.items():
    mean, _ = tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=DETAIL_TOP, tiebreak=TIEBREAK)
    display(Markdown(f"### {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}"))
    display(tracking.style_best_of(mean))

### Coefficients

Averaged over the windows. The first table is each block's base. The second shows the coefficients of the terms each feature set adds. For a fixed block, `inc_signed` is the imposed value. A `pres_party_signed:<variable>` coefficient is the effect on the president's party's margin per unit of the variable.

In [ ]:
_, COEFFICIENTS = tracking.rolling_best_of(rolling, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)
COEFFICIENTS.xs("base", level="run").round(3)

In [ ]:
# the added terms' coefficients: feature set x block
added = pd.DataFrame({
    block: {name: ", ".join(f"{term} {COEFFICIENTS.loc[(block, name), term]:+.3f}" for term in terms)
            for name, terms in FEATURE_SETS.items() if terms}
    for block in BLOCKS
})
added.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

## 4. Model B without partisan lean in the ensemble


Model B, the fundamentals model blended at 15% into `house_ensemble_ab_85_15`, includes `partisan_lean`, which Model A already uses. This notebook asks whether a **lean-free B** adds information that complements A better. It compares:

| model | prediction | role |
|---|---|---|
| **A** | Model A (polling), unchanged: `partisan_lean + gen_margin_raw + inc_signed` (fixed +2.5) `+ partisan_lean:inc_signed + gb_momentum_30d_raw` | reference |
| **AB-15** | 0.85 A + 0.15 B; B = `partisan_lean + fundraising_advantage + pres_party_signed:is_midterm + pres_party_signed:cpi_region_log_yoy` | baseline (`house_ensemble_ab_85_15`) |
| **AB-NoLean-15** | 0.85 A + 0.15 B-NoLean; B-NoLean = `fundraising_advantage + rel_state_college + pres_party_signed:is_midterm + pres_party_signed:cpi_region_log_yoy` | alternative |
| **Shrink-15** | A − 0.15 × (A's partisan-lean terms) = 0.85 A + 0.15 (A without its lean terms) | diagnostic only |

**Shrink-15** is a benchmark for one specific explanation. Blending A at 0.85 with a lean-free model scales A's lean contribution (`partisan_lean` and `partisan_lean:inc_signed`) by 0.85. Shrink-15 applies exactly that shrinkage and nothing else: no fundraising, no college, no national fundamentals. If AB-NoLean-15 does no better than Shrink-15, its gain is structural shrinkage rather than new information.

**Design:**

- **Fixed specification:** the two established configurations in `house.stage1` are used as is. B-NoLean is defined here only, the blend weight stays fixed at 0.15, and nothing is tuned or selected.
- **Estimation:** every model is fit by equal-year WLS (`stage1.fit_predict`), on expanding windows from 2006 through the election before each eval year.
- **Matched samples:** in each window all models fit and score exactly the same district-years (asserted).
- **Evaluation years:** 2018, 2020, 2022, and 2024. **2024 is not an untouched holdout here:** `01_reg_final_2024_holdout.ipynb` already scored it, so it is development evidence like the other years.
- **Subsets:** valid targets (observed, non-imputed, D-vs-R race types), ex-ante competitive (|lean + eve-of-election generic ballot| ≤ 10), and ex-post competitive (|actual two-party margin| ≤ 7.5).
- **Residuals:** actual − predicted, in D − R points. Positive means Democrats outperformed the model.
- **Averages:** *pooled* averages weight every race equally; *equal-year* averages are the mean of the four per-year values.

> **Re-run note (2026-11-03).** This section now runs with the current Model A, which dropped `gb_momentum_30d_raw` on 2026-11-02. Its prose and the figures it quotes were written for the earlier Model A with momentum, so the tables below differ slightly. The conclusion holds: pooled 2018–2024 MAE for AB-15 vs AB-NoLean-15 is 5.17 vs 4.63 (all districts) and 4.68 vs 4.51 (ex-ante competitive), against 5.07 vs 4.53 and 4.57 vs 4.41 before. Most of the gain is still from offsetting structural errors in safe seats, so AB-15 is kept.

### Settings and models

In [ ]:
TARGET = stage1.TARGET
EVAL_YEARS = [2018, 2020, 2022, 2024]  # each fit on 2006 through the election before it
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
EX_ANTE_THRESHOLD = 10
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),  # known before the election
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),                     # the observed margin
}
SECTIONS = {"EX_ANTE": "competitive_ex_ante", "EX_POST": "competitive_ex_post", "ALL": "valid"}  # priority order
WEIGHT_B = 0.15  # fixed; not tuned

# The established configurations, used unchanged
assert stage1.HOUSE_ENSEMBLE_AB_85_15.components == ((stage1.MODEL_A, 0.85), (stage1.MODEL_B, 0.15))
assert stage1.HOUSE_MODEL_A.components == ((stage1.MODEL_A, 1.0),)

# The alternative, defined here only: B without partisan_lean, with within-state relative college share
MODEL_B_NOLEAN = stage1.Component(
    "B_nolean",
    f"{TARGET} ~ fundraising_advantage + rel_state_college + pres_party_signed:is_midterm"
    " + pres_party_signed:cpi_region_log_yoy",
)
AB_NOLEAN_15 = stage1.Stage1Config("ab_nolean_15", ((stage1.MODEL_A, 1 - WEIGHT_B), (MODEL_B_NOLEAN, WEIGHT_B)))
CONFIGS = {"AB-15": stage1.HOUSE_ENSEMBLE_AB_85_15, "AB-NoLean-15": AB_NOLEAN_15}
assert "partisan_lean" not in MODEL_B_NOLEAN.formula

for label, config in CONFIGS.items():
    print(label, [(c.name, w) for c, w in config.components])
    for component, _ in config.components:
        print(f"    {component.name:15s} {component.formula}   fixed: {dict(component.fixed) or 'none'}")

### Rolling fits

Both ensembles are fit in every window. Model A appears in each, so its predictions must agree, and every component must use the same training and evaluation rows. Both conditions are asserted.

In [ ]:
with warnings.catch_warnings(action="ignore"):
    rolling = {label: stage1.rolling_predictions(config, EVAL_YEARS, predictors=predictors, targets=targets)
               for label, config in CONFIGS.items()}

for year in EVAL_YEARS:
    base, alt = rolling["AB-15"][year], rolling["AB-NoLean-15"][year]
    runs = [*base.components.values(), *alt.components.values()]
    assert len({(r.counts["train_rows_hash"], r.counts["eval_rows_hash"]) for r in runs}) == 1, year
    assert base.predictions[KEY].equals(alt.predictions[KEY]), year
    assert np.array_equal(base.predictions["prediction_A_polling"], alt.predictions["prediction_A_polling"]), year
    print(year, "trained on", runs[0].config["train_years"][0], "-", runs[0].config["train_years"][-1],
          "| train rows", runs[0].counts["n_train"], "| eval rows", len(base.predictions))

### The shrinkage-only benchmark

In each window, A's lean contribution for a district is $\hat\beta_P P + \hat\beta_{PI}\,(P\cdot I)$, with A's own fitted coefficients for that window. Shrink-15 subtracts 15% of it from A's prediction and changes nothing else.

In [ ]:
def lean_contribution(result, rows):
    # Model A's partisan-lean terms for `rows`, with the window's fitted A coefficients
    params = result.components["A_polling"].model.params
    return (params["partisan_lean"] * rows["partisan_lean"]
            + params["partisan_lean:inc_signed"] * rows["partisan_lean"] * rows["inc_signed"])


MODELS = ["A", "AB-15", "AB-NoLean-15", "Shrink-15", "B", "B-NoLean"]
ENSEMBLES = ["AB-15", "AB-NoLean-15", "Shrink-15"]
frames = []
for year in EVAL_YEARS:
    base, alt = rolling["AB-15"][year].predictions, rolling["AB-NoLean-15"][year].predictions
    rows = base[KEY].merge(predictors, on=KEY, how="left", validate="one_to_one")
    frames.append(base[KEY].assign(**{
        "A": base["prediction_A_polling"],
        "AB-15": base["prediction"],
        "AB-NoLean-15": alt["prediction"],
        "Shrink-15": base["prediction_A_polling"] - WEIGHT_B * lean_contribution(rolling["AB-15"][year], rows),
        "B": base["prediction_B_fundamentals"],
        "B-NoLean": alt["prediction_B_nolean"],
    }))
predictions = pd.concat(frames, ignore_index=True)

# Score every model like a single model, on the valid targets
scores = {m: evaluation.evaluate_predictions(predictions[KEY].assign(prediction=predictions[m]), targets, predictors,
                                             TARGET, competitive=COMPETITIVE, race_types=EVAL_RACE_TYPES)
          for m in MODELS}
metrics = {m: s[0] for m, s in scores.items()}
scored = {m: s[2] for m, s in scores.items()}
assert all(scored[m][KEY].equals(scored["A"][KEY]) for m in MODELS)  # identical valid district-years

# One row per valid district-year; PRED and RESID hold each model's prediction and residual on the same rows
races = scored["A"][[*KEY, "actual_margin", "competitive_ex_ante", "competitive_ex_post"]].merge(
    predictors[[*KEY, "partisan_lean", "inc_signed"]], on=KEY, how="left", validate="one_to_one")
PRED = pd.DataFrame({m: scored[m]["predicted_margin"].to_numpy() for m in MODELS}, index=races.index)
RESID = races["actual_margin"].to_numpy()[:, None] - PRED
MASKS = {"EX_ANTE": races["competitive_ex_ante"], "EX_POST": races["competitive_ex_post"],
         "ALL": pd.Series(True, index=races.index)}
scores["A"][1]  # coverage: districts per eval year by exclusion reason, and the subset sizes

### 1. Performance

MAE and winner accuracy for each subset, per eval year, pooled over the four years, and as the equal-year mean. Δ is model − A on the same races. ✓ marks an improvement over A and ✗ a regression. Shrink-15 is diagnostic only. B and B-NoLean are the standalone fundamentals models, never used alone.

In [ ]:
METRICS = ["n", "mae", "rmse", "accuracy_pct", "missed_races"]
DELTAS = ["mae", "accuracy_pct", "missed_races"]


def performance(period):
    # model x (section, metric); period is an eval year, "pooled", or "equal-year mean"
    def values(m, subset):
        if period == "pooled":
            return metrics[m].loc[(subset, "all")]
        if period == "equal-year mean":
            return sum(metrics[m].loc[(subset, y)] for y in EVAL_YEARS) / len(EVAL_YEARS)
        return metrics[m].loc[(subset, period)]

    rows = {}
    for m in MODELS:
        row = {}
        for section, subset in SECTIONS.items():
            v, a = values(m, subset), values("A", subset)
            row |= {(section, k): v[k] for k in METRICS}
            if m != "A":
                row |= {(section, f"Δ{k}"): v[k] - a[k] for k in DELTAS}
        rows[m] = row
    return pd.DataFrame.from_dict(rows, orient="index")


for period in ["pooled", "equal-year mean"]:
    display(Markdown(f"### {period.capitalize()}, 2018–2024"))
    display(tracking.style_best_of(performance(period)))

In [ ]:
by_year = pd.concat({year: performance(year) for year in EVAL_YEARS}, names=["eval_year", "model"]).swaplevel()
by_year = by_year.reindex([(m, y) for m in MODELS for y in EVAL_YEARS])
display(Markdown("### Per eval year"))
tracking.style_best_of(by_year)

### 2. Winner calls fixed and broken relative to A

A call is **fixed** when the model is right and A was wrong, and **broken** when A was right and the model is wrong. Net = fixed − broken: the change in correct calls. A race counts in every subset it belongs to.

In [ ]:
def correct(m):
    return pd.Series(np.sign(PRED[m]) == np.sign(races["actual_margin"]), index=races.index)


flip_rows = {}
for m in ENSEMBLES + ["B", "B-NoLean"]:
    fixed, broken = correct(m) & ~correct("A"), ~correct(m) & correct("A")
    for section, mask in MASKS.items():
        for year in [*EVAL_YEARS, "pooled"]:
            in_year = mask if year == "pooled" else mask & (races["year"] == year)
            f, b = int((fixed & in_year).sum()), int((broken & in_year).sum())
            flip_rows[(m, year)] = flip_rows.get((m, year), {}) | {(section, "fixed"): f, (section, "broken"): b,
                                                                    (section, "net"): f - b}
flips = pd.DataFrame.from_dict(flip_rows, orient="index").rename_axis(["model", "eval_year"])
flips.style.map(lambda v: f"color: {tracking.GOOD}" if v > 0 else f"color: {tracking.BAD}" if v < 0 else "",
                subset=[c for c in flips.columns if c[1] == "net"])

In [ ]:
# Every competitive race where an ensemble's call differs from A's
competitive = races["competitive_ex_ante"] | races["competitive_ex_post"]
changed = []
for m in ENSEMBLES:
    differs = competitive & (np.sign(PRED[m]) != np.sign(PRED["A"]))
    for i in races.index[differs]:
        r = races.loc[i]
        changed.append({"model": m, "year": r["year"], "district": evaluation.district_label(r["state"], r["district"]),
                        "ex_ante": r["competitive_ex_ante"], "ex_post": r["competitive_ex_post"],
                        "lean": r["partisan_lean"], "actual": r["actual_margin"], "A": PRED.loc[i, "A"],
                        "model_pred": PRED.loc[i, m], "verdict": "fixed" if correct(m)[i] else "broken"})
changed = pd.DataFrame(changed)
changed.style.format({"lean": "{:+.1f}", "actual": "{:+.1f}", "A": "{:+.1f}", "model_pred": "{:+.1f}"}).map(
    lambda v: f"color: {tracking.GOOD}" if v == "fixed" else f"color: {tracking.BAD}" if v == "broken" else "",
    subset=["verdict"])

### 3. Standalone Model B and coefficient stability

The coefficients in each window (coef, with the WLS standard error beneath in the second table), and their spread across the four windows. A stable fundamentals model keeps its signs and magnitudes as each election is added.

In [ ]:
COMPONENTS = {"A": ("AB-15", "A_polling"), "B": ("AB-15", "B_fundamentals"), "B-NoLean": ("AB-NoLean-15", "B_nolean")}


def coefficient_table(model, column):
    label, name = COMPONENTS[model]
    return pd.DataFrame({year: rolling[label][year].components[name].coefficients[column] for year in EVAL_YEARS})


coefs = pd.concat({m: coefficient_table(m, "coef") for m in COMPONENTS}, names=["model", "term"])
spread = pd.DataFrame({"mean": coefs.mean(axis=1), "sd": coefs.std(axis=1), "min": coefs.min(axis=1),
                       "max": coefs.max(axis=1)})
spread["sd/|mean|"] = spread["sd"] / spread["mean"].abs()
spread["sign changes"] = (np.sign(coefs).nunique(axis=1) > 1)
display(Markdown("### Coefficients by eval window, with their spread"))
display(coefs.join(spread).style.format(precision=3))
display(Markdown("### Standard errors by eval window (blank: fixed coefficient)"))
pd.concat({m: coefficient_table(m, "std_err") for m in COMPONENTS}, names=["model", "term"]).round(3)

### 4. Residual distributions

Pooled out-of-sample residuals (actual − predicted) for each subset. Skewness and excess kurtosis are 0 for a normal distribution. The `>10`, `>15`, and `>20` columns are the percentage of races whose absolute error exceeds that many points.

In [ ]:
def residual_stats(r):
    r = pd.Series(r).astype(float)
    q = r.quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    return {"n": len(r), "bias": r.mean(), "mae": r.abs().mean(), "rmse": np.sqrt((r ** 2).mean()), "sd": r.std(),
            "q05": q[0.05], "q25": q[0.25], "median": q[0.5], "q75": q[0.75], "q95": q[0.95],
            "skew": stats.skew(r), "excess_kurtosis": stats.kurtosis(r),
            **{f">{t}": 100 * (r.abs() > t).mean() for t in (10, 15, 20)}}


distribution = pd.DataFrame.from_dict({(section, m): residual_stats(RESID.loc[mask, m])
                                       for section, mask in MASKS.items() for m in MODELS}, orient="index")
distribution.index.names = ["subset", "model"]
distribution.style.format(precision=2).format({"n": "{:.0f}"})

In [ ]:
COLORS = {"A": "#2a78d6", "AB-15": "#eb6834", "AB-NoLean-15": "#1baf7a", "Shrink-15": "#eda100"}
STYLES = {"A": "-", "AB-15": "-", "AB-NoLean-15": "-", "Shrink-15": "--"}


def tidy(ax, title, xlabel, ylabel):
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors="#52514e")


fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
thresholds = np.arange(0, 30.5, 0.5)
for ax, section in zip(axes, ["ALL", "EX_ANTE"]):
    for m in COLORS:
        r = RESID.loc[MASKS[section], m].abs()
        ax.plot(thresholds, [100 * (r > t).mean() for t in thresholds], STYLES[m], color=COLORS[m], lw=2, label=m)
    ax.set_yscale("log")
    tidy(ax, f"Share of races missed by more than x points ({section.lower()})", "x (points)", "% of races (log)")
axes[0].legend(frameon=False)
plt.tight_layout()

### 5. Residual correlation: A vs each Model B

A blend helps most when its components' errors are weakly correlated. The pooled correlations include each year's national error, which every model shares. The *within-year* correlations remove each year's mean residual first.

In [ ]:
def correlations(rows, within_year):
    r = RESID.loc[rows, ["A", "B", "B-NoLean"]]
    if within_year:
        r = r - r.groupby(races.loc[rows, "year"]).transform("mean")
    return {"n": len(r), "corr(A, B)": r["A"].corr(r["B"]), "corr(A, B-NoLean)": r["A"].corr(r["B-NoLean"]),
            "corr(B, B-NoLean)": r["B"].corr(r["B-NoLean"])}


corr_rows = {}
for section, mask in MASKS.items():
    for year in EVAL_YEARS:
        corr_rows[(section, year)] = correlations(mask & (races["year"] == year), within_year=False)
    corr_rows[(section, "equal-year mean")] = pd.DataFrame([corr_rows[(section, y)] for y in EVAL_YEARS]).mean().to_dict()
    corr_rows[(section, "pooled")] = correlations(mask, within_year=False)
    corr_rows[(section, "pooled, within-year")] = correlations(mask, within_year=True)
pd.DataFrame.from_dict(corr_rows, orient="index").rename_axis(["subset", "period"]).style.format(precision=3).format(
    {"n": "{:.0f}"})

### 6. Residuals vs partisan lean

A model that systematically under- or over-weights the partisan baseline leaves residuals that trend with `partisan_lean`.

- **Signed bins:** the mean residual by lean bin, from strongly R (left) to strongly D (right).
- **Magnitude bins:** the mean of residual × sign(lean), by |lean|. A positive value means the model predicted the district *too close to even* (too little lean), and a negative value *too far from even*.
- **Slopes:** OLS of the residual on `partisan_lean` with eval-year fixed effects, per 10 points of lean (standard errors are not clustered).
- **Effective lean slope:** the same regression with the *prediction* as the outcome, relative to A's. It measures how much of A's partisan structure each model keeps.

In [ ]:
LEAN_BINS = [-np.inf, -30, -15, -5, 5, 15, 30, np.inf]
LEAN_LABELS = ["R+30<", "R+15–30", "R+5–15", "±5", "D+5–15", "D+15–30", "D+30<"]
MAGNITUDE_BINS = [0, 5, 15, 30, np.inf]
MAGNITUDE_LABELS = ["0–5", "5–15", "15–30", "30+"]

signed_bin = pd.cut(races["partisan_lean"], LEAN_BINS, labels=LEAN_LABELS)
signed = RESID.groupby(signed_bin, observed=True).mean()
signed.insert(0, "n", signed_bin.value_counts().reindex(signed.index))
display(Markdown("### Mean residual by signed lean bin (all valid races)"))
display(signed.style.format(precision=2).format({"n": "{:.0f}"}))

magnitude_bin = pd.cut(races["partisan_lean"].abs(), MAGNITUDE_BINS, labels=MAGNITUDE_LABELS, right=False)
toward_even = RESID.mul(np.sign(races["partisan_lean"]), axis=0)
magnitude = pd.concat({"mean residual × sign(lean)": toward_even.groupby(magnitude_bin, observed=True).mean(),
                       "MAE": RESID.abs().groupby(magnitude_bin, observed=True).mean()}, axis=1)
magnitude.insert(0, ("n", ""), magnitude_bin.value_counts().reindex(magnitude.index))
display(Markdown("### By |lean|: residual toward the lean (+ = predicted too close to even), and MAE"))
display(magnitude.style.format(precision=2).format({("n", ""): "{:.0f}"}))


def lean_slope(outcome, rows):
    fit = smf.ols("y ~ partisan_lean + C(year)", data=races.loc[rows].assign(y=outcome[rows])).fit()
    return 10 * fit.params["partisan_lean"], 10 * fit.bse["partisan_lean"]


slopes = {}
for section, mask in MASKS.items():
    for m in MODELS:
        resid_slope, resid_se = lean_slope(RESID[m], mask)
        pred_slope, _ = lean_slope(PRED[m], mask)
        slopes[(section, m)] = {"residual slope per 10 pts": resid_slope, "se": resid_se,
                                "prediction slope per 10 pts": pred_slope}
slopes = pd.DataFrame.from_dict(slopes, orient="index").rename_axis(["subset", "model"])
slopes["effective lean vs A"] = slopes["prediction slope per 10 pts"] / slopes.xs("A", level="model")[
    "prediction slope per 10 pts"].reindex(slopes.index.get_level_values("subset")).to_numpy()
display(Markdown("### Residual and prediction slopes on partisan lean (year fixed effects)"))
slopes.style.format(precision=3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
x = np.arange(len(LEAN_LABELS))
for m in COLORS:
    axes[0].plot(x, signed[m], STYLES[m], color=COLORS[m], lw=2, marker="o", ms=8, label=m)
axes[0].set_xticks(x, LEAN_LABELS)
axes[0].axhline(0, color="#8d8c86", lw=1)
tidy(axes[0], "Mean residual by partisan lean (all valid races)", "partisan_lean bin", "mean actual − predicted (pts)")
axes[0].legend(frameon=False)
x = np.arange(len(MAGNITUDE_LABELS))
for m in COLORS:
    axes[1].plot(x, magnitude[("mean residual × sign(lean)", m)], STYLES[m], color=COLORS[m], lw=2, marker="o", ms=8)
axes[1].set_xticks(x, MAGNITUDE_LABELS)
axes[1].axhline(0, color="#8d8c86", lw=1)
tidy(axes[1], "Residual toward the lean, by |lean| (+ = too close to even)", "|partisan_lean|", "points")
plt.tight_layout()

### 7. Residuals by election year and state

- **National directional error:** each year's mean residual, shared by every district that year.
- **Decomposition:** the mean squared residual splits exactly into three parts. *National* is the year mean², *state* is the state-year mean minus the year mean, squared, and *district* is the residual minus its state-year mean, squared. All are averaged over races. At-large states have one district per year, so their deviations fall entirely in *state*.
- **States:** the states, with at least 12 scored district-years, whose mean residual moves most between AB-15 and AB-NoLean-15.

In [ ]:
yearly = pd.concat({"mean residual (bias)": RESID.groupby(races["year"]).mean(),
                    "MAE": RESID.abs().groupby(races["year"]).mean()}, axis=1)
display(Markdown("### By eval year (all valid races)"))
display(yearly.style.format(precision=2))
yearly_bias = pd.concat({section: RESID[mask].groupby(races.loc[mask, "year"]).mean()[["A", "AB-15", "AB-NoLean-15",
                                                                                      "Shrink-15", "B-NoLean"]]
                         for section, mask in MASKS.items()}, axis=1)
display(Markdown("### Mean residual (bias) by eval year and subset"))
display(yearly_bias.style.format(precision=2))

year_mean = RESID.groupby(races["year"]).transform("mean")
state_year_mean = RESID.groupby([races["year"], races["state"]]).transform("mean")
decomposition = pd.DataFrame({"MSE": (RESID ** 2).mean(), "national": (year_mean ** 2).mean(),
                              "state": ((state_year_mean - year_mean) ** 2).mean(),
                              "district": ((RESID - state_year_mean) ** 2).mean()})
assert np.allclose(decomposition["MSE"], decomposition[["national", "state", "district"]].sum(axis=1))
for part in ["national", "state", "district"]:
    decomposition[f"{part} %"] = 100 * decomposition[part] / decomposition["MSE"]
display(Markdown("### Mean squared residual: national, state, and district parts (all valid races)"))
display(decomposition.style.format(precision=2))

state_n = races.groupby("state").size()
state_mean = RESID.groupby(races["state"]).mean()[state_n >= 12]
state_mean["n"] = state_n
state_mean["NoLean − AB-15"] = state_mean["AB-NoLean-15"] - state_mean["AB-15"]
display(Markdown("### States whose mean residual moves most from AB-15 to AB-NoLean-15 (2018–2024)"))
state_mean.reindex(state_mean["NoLean − AB-15"].abs().sort_values(ascending=False).index).head(10)[
    ["n", "A", "AB-15", "AB-NoLean-15", "Shrink-15", "NoLean − AB-15"]].style.format(precision=2).format({"n": "{:.0f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for m in COLORS:
    ax.plot(EVAL_YEARS, yearly[("mean residual (bias)", m)], STYLES[m], color=COLORS[m], lw=2, marker="o", ms=8, label=m)
ax.set_xticks(EVAL_YEARS)
ax.axhline(0, color="#8d8c86", lw=1)
tidy(ax, "National directional error: mean residual by year", "eval year", "mean actual − predicted (pts)")
ax.legend(frameon=False)
plt.tight_layout()

### 8. Where does the gain come from: safe or competitive races?

Valid races are split into three exclusive groups: ex-ante competitive, ex-post competitive only (a surprise close race), and safe (neither). For each ensemble, the table shows the change in MAE versus A within each group, and each group's share of the total change in absolute error (pooled, 2018–2024). A negative change is an improvement.

In [ ]:
group = np.select([races["competitive_ex_ante"], races["competitive_ex_post"]],
                  ["ex-ante competitive", "ex-post only"], "safe")
abs_change = RESID[ENSEMBLES].abs().sub(RESID["A"].abs(), axis=0)
gain = {}
for m in ENSEMBLES:
    total = abs_change[m].sum()
    for g in ["ex-ante competitive", "ex-post only", "safe"]:
        rows = group == g
        gain[(m, g)] = {"n": int(rows.sum()), "MAE A": RESID.loc[rows, "A"].abs().mean(),
                        "MAE model": RESID.loc[rows, m].abs().mean(), "ΔMAE": abs_change.loc[rows, m].mean(),
                        "Σ Δ|residual|": abs_change.loc[rows, m].sum(),
                        "share of total Δ (%)": 100 * abs_change.loc[rows, m].sum() / total}
pd.DataFrame.from_dict(gain, orient="index").rename_axis(["model", "group"]).style.format(precision=2).format(
    {"n": "{:.0f}"})

#### Paired comparison of AB-NoLean-15 with AB-15 and Shrink-15

On the same races: the mean change in absolute error (negative favors AB-NoLean-15), its naive standard error (races treated as independent, so it is optimistic), the share of races where AB-NoLean-15 is closer, and the number of eval years (of 4) whose MAE favors it.

In [ ]:
paired = {}
for other in ["AB-15", "Shrink-15"]:
    diff = RESID["AB-NoLean-15"].abs() - RESID[other].abs()
    for section, mask in MASKS.items():
        d = diff[mask]
        per_year = d.groupby(races.loc[mask, "year"]).mean()
        paired[(other, section)] = {"n": len(d), "mean Δ|residual|": d.mean(), "naive se": d.std() / np.sqrt(len(d)),
                                    "% races closer": 100 * (d < 0).mean(), "years favoring NoLean": int((per_year < 0).sum())}
pd.DataFrame.from_dict(paired, orient="index").rename_axis(["vs", "subset"]).style.format(precision=3).format(
    {"n": "{:.0f}", "years favoring NoLean": "{:.0f}", "% races closer": "{:.1f}"})

### 9. Shrinkage or complementary information?

AB-NoLean-15 moves A's prediction by 0.15 × (B-NoLean − A). Shrink-15 moves it by −0.15 × (A's lean terms). If the first adjustment is mostly the second, a regression of one on the other has slope ≈ 1 and a high R², and whatever is left over is what the fundamentals add. The same regression is shown for AB-15. The last column checks whether the leftover part helps: it is the MAE of A plus the leftover alone, without the shrinkage.

In [ ]:
shrink_move = PRED["Shrink-15"] - PRED["A"]
rows = {}
for m in ["AB-15", "AB-NoLean-15"]:
    move = PRED[m] - PRED["A"]
    for section, mask in MASKS.items():
        fit = smf.ols("move ~ shrink", data=pd.DataFrame({"move": move[mask], "shrink": shrink_move[mask]})).fit()
        leftover = fit.resid + fit.params["Intercept"]  # the part of the move not proportional to the shrinkage
        rows[(m, section)] = {"slope on shrink move": fit.params["shrink"], "R²": fit.rsquared,
                              "sd(move)": move[mask].std(), "sd(leftover)": leftover.std(),
                              "MAE A": RESID.loc[mask, "A"].abs().mean(),
                              "MAE A + leftover": (races.loc[mask, "actual_margin"] - PRED.loc[mask, "A"] - leftover).abs().mean(),
                              "MAE model": RESID.loc[mask, m].abs().mean()}
pd.DataFrame.from_dict(rows, orient="index").rename_axis(["model", "subset"]).style.format(precision=3)

### Conclusion

**Verdict:** removing partisan lean from B mostly corrects a structural flaw in Model A, adds a small amount of competitive-race information, and introduces a Democratic-ward bias in competitive races. AB-NoLean-15 should **not replace AB-15**. The diagnostic worth following up is A's over-extension in safe seats, not the lean-free B.

**1. Headline numbers.** AB-NoLean-15 beats AB-15 in every subset and every eval year (pooled MAE):

| subset | A | AB-15 | AB-NoLean-15 |
|---|---|---|---|
| ex-ante competitive | 4.85 | 4.57 | 4.41 |
| ex-post competitive | 6.18 | 5.74 | 5.33 |
| all districts | 5.36 | 5.07 | 4.53 |

The equal-year averages tell the same story. On winner calls, AB-NoLean-15 nets +4 against A (8 fixed, 4 broken), while AB-15 nets +2. That is two more correct calls in four elections. Three of AB-NoLean-15's four net gains come from 2018, and it nets 0 in both 2022 and 2024.

**2. Most of the gain is in safe seats.** About 81% of the total reduction in absolute error comes from safe districts; ex-ante competitive races supply 12%.

- **A over-extends lopsided districts.** Its residual toward the lean is −4.7 points where |lean| ≥ 30: it predicts them too far from even.
- **B-NoLean errs the opposite way** (+9.5 points: far too close to even), so the two errors offset.
- That offset is what produces the negative all-district residual correlation between A and B-NoLean (−0.25), against +0.41 for A and B. It is an offset of opposite structural errors, not independent information.

**3. Shrinkage explains much of the gain, but not all of it.**

- **Ex-post competitive: pure shrinkage.** Shrink-15 does as well as AB-NoLean-15 (MAE 5.23 vs 5.33; AB-NoLean-15 is worse in 2 of 4 years). The shrinkage move explains 62% of AB-NoLean-15's move (slope 0.91), and the leftover part doesn't improve MAE at all (6.18 → 6.18).
- **All districts: partly shrinkage.** Shrink-15 recovers about half of the gain (−0.46 of −0.83). The rest is a nonlinear correction concentrated in the safest seats, which uniform linear shrinkage can't mimic: Shrink-15 overshoots there and flips the residual's sign in both 30+ bins.
- **Ex-ante competitive: not shrinkage.** Shrink-15 gains almost nothing here (−0.03), the shrinkage move explains only 6% of AB-NoLean-15's move, and the leftover carries the whole gain (4.85 → 4.44). This is the only real evidence of complementary information. Against AB-15 it is −0.16 MAE (naive standard error 0.06), it holds in 4 of 4 years, and AB-NoLean-15 is closer in 55% of races.

**4. The competitive-race gain comes with a Democratic-ward bias.** A negative residual means the model was too Democratic.

- **Pooled bias:** in ex-ante competitive races AB-NoLean-15's bias is −1.46, against −0.67 for AB-15 and −0.47 for A; in ex-post races it is −1.19 against −0.34.
- **By year:** AB-NoLean-15 is more Democratic-biased than AB-15 in the ex-ante competitive races of 2020, 2022, and 2024. The shift helped only in 2018, when A was too Republican.
- **Calls:** every competitive call it changed moved from R to D.
- **Lean bins:** residuals in the ±5 and D+5–15 bins become more negative.
- **B-NoLean alone is not a credible fundamentals model:**
  - its ex-ante bias is −7.1 and its MAE is 9.2 (12.1 over all districts)
  - its inflation coefficient flips sign relative to B (+0.12 vs −0.44)
  - its midterm penalty is unstable: −3.4 in the 2018 window, then about −1.3
  - its value comes only from offsetting A's errors

**5. Tails and dispersion.** AB-NoLean-15 has fewer large errors:

- **Misses over 10 points:** 8.1% of races, against 11.4% for AB-15 and 13.8% for A.
- **Misses over 20 points:** 0.56%, against 0.94% for AB-15.

Skewness and excess kurtosis barely change. Compared with AB-15, the reduction is in the state and district parts of the squared error; the national part is unchanged.

**Recommendation.** Keep AB-15 (`house_ensemble_ab_85_15`) as the Stage 1 ensemble.

- **Why not switch:** AB-NoLean-15's edge in competitive races is small (−0.16 ex-ante MAE and 2 calls in four elections, mostly 2018). It comes with a systematic Democratic-ward shift in competitive races, and through a component that is not interpretable on its own.
- **What to follow up instead:** the larger and cleaner finding is that Model A over-extends partisan lean in safe districts. That belongs in a separate, pre-specified check of A's lean specification, which could capture the safe-seat gain without B-NoLean's bias.
- **When to revisit:** only if the no-lean ex-ante gain persists once A's safe-seat over-extension is addressed.
- **Evidence base:** four elections, all previously examined (including 2024), so every difference here is development evidence, not a confirmatory test.